# B 站视频对话提取

从 B 站视频获取对话文本，并按**不同说话人分段**输出。

- 优先尝试 B 站字幕（可配置 `SESSDATA` 登录 Cookie）
- 无字幕时：下载音频 → 说话人聚类 → Whisper 转写 → 按人合并文本

默认视频：[BV18ijD6BExE](https://www.bilibili.com/video/BV18ijD6BExE)

**依赖安装：**

```bash
pip install imageio-ffmpeg librosa scikit-learn openai-whisper torch
```

输出目录：`data/video_dialogues/`（生成 `.txt` 与 `.json`）

In [ ]:
import json
import os
import re
import subprocess
import tempfile
import time
import urllib.error
import urllib.parse
import urllib.request
from collections import defaultdict
from pathlib import Path

import imageio_ffmpeg
import librosa
import numpy as np
import whisper
from sklearn.cluster import AgglomerativeClustering

# ========== 配置 ==========
VIDEO_URL = (
    "https://www.bilibili.com/video/BV18ijD6BExE/"
    "?spm_id_from=333.1387.collection.video_card.click"
)
# 可选：登录后从浏览器复制 SESSDATA，用于读取 B 站 AI/CC 字幕
SESSDATA = os.environ.get("BILIBILI_SESSDATA", "")

MAX_DURATION_SEC = None  # 设为整数可只处理前 N 秒（调试）；None 处理全片
WHISPER_MODEL = "base"  # tiny/base/small/medium，越大越准越慢
N_SPEAKERS = 12         # 说话人聚类上限（狼人杀局内可设 12）
REQUEST_INTERVAL_SEC = 0.4

OUTPUT_DIR = Path("data") / "video_dialogues"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://www.bilibili.com",
}
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

BVID = re.search(r"BV[\w]+", VIDEO_URL).group(0)
print("BVID:", BVID)

In [ ]:
def fetch_json(url: str, cookie: str = "") -> dict:
    headers = dict(HEADERS)
    if cookie:
        headers["Cookie"] = cookie
    last_error = None
    for attempt in range(1, 4):
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=60) as resp:
                payload = json.loads(resp.read().decode("utf-8"))
            if payload.get("code") != 0:
                raise RuntimeError(payload.get("message") or payload)
            time.sleep(REQUEST_INTERVAL_SEC)
            return payload["data"]
        except (urllib.error.URLError, TimeoutError, RuntimeError) as exc:
            last_error = exc
            time.sleep(REQUEST_INTERVAL_SEC * attempt)
    raise last_error


def parse_bvid(url: str) -> str:
    match = re.search(r"BV[\w]+", url)
    if not match:
        raise ValueError(f"无法解析 bvid: {url}")
    return match.group(0)


def get_video_meta(bvid: str) -> dict:
    data = fetch_json(f"https://api.bilibili.com/x/web-interface/view?bvid={bvid}")
    return {
        "bvid": bvid,
        "aid": data["aid"],
        "cid": data["cid"],
        "title": data.get("title", bvid),
        "duration": int(data.get("duration") or 0),
    }


def get_cookie_header() -> str:
    return f"SESSDATA={SESSDATA}" if SESSDATA else ""


def list_subtitle_tracks(bvid: str, cid: int) -> list[dict]:
    cookie = get_cookie_header()
    data = fetch_json(
        f"https://api.bilibili.com/x/player/wbi/v2?bvid={bvid}&cid={cid}",
        cookie=cookie,
    )
    return (data.get("subtitle") or {}).get("subtitles") or []


def download_subtitle_json(subtitle_url: str) -> dict:
    url = subtitle_url if subtitle_url.startswith("http") else "https:" + subtitle_url
    cookie = get_cookie_header()
    return fetch_json(url, cookie=cookie) if subtitle_url.endswith(".json") else json.loads(
        urllib.request.urlopen(
            urllib.request.Request(url, headers={**HEADERS, "Cookie": cookie})
        ).read().decode("utf-8")
    )


def get_audio_stream_url(bvid: str, cid: int) -> str:
    data = fetch_json(
        f"https://api.bilibili.com/x/player/playurl?bvid={bvid}&cid={cid}&fnval=16"
    )
    audio = (data.get("dash") or {}).get("audio") or []
    if not audio:
        raise RuntimeError("未获取到音频流")
    return audio[0]["baseUrl"]


def download_audio_wav(
    bvid: str,
    cid: int,
    out_wav: Path,
    max_duration_sec: int | None = None,
) -> Path:
    audio_url = get_audio_stream_url(bvid, cid)
    raw_m4s = out_wav.with_suffix(".m4s")
    req = urllib.request.Request(
        audio_url,
        headers={**HEADERS, "Referer": f"https://www.bilibili.com/video/{bvid}"},
    )
    with urllib.request.urlopen(req, timeout=300) as resp, raw_m4s.open("wb") as f:
        while True:
            chunk = resp.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)

    cmd = [FFMPEG, "-y", "-i", str(raw_m4s), "-ac", "1", "-ar", "16000"]
    if max_duration_sec:
        cmd.extend(["-t", str(max_duration_sec)])
    cmd.append(str(out_wav))
    subprocess.run(cmd, check=True, capture_output=True)
    raw_m4s.unlink(missing_ok=True)
    return out_wav

In [ ]:
def format_ts(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    if h:
        return f"{h:02d}:{m:02d}:{s:05.2f}"
    return f"{m:02d}:{s:05.2f}"


def subtitle_entries_from_bilibili(payload: dict) -> list[dict]:
    body = payload.get("body") or []
    entries = []
    for item in body:
        text = str(item.get("content", "")).strip()
        if not text:
            continue
        sid = item.get("sid")
        speaker = item.get("speaker")
        if speaker:
            label = str(speaker).strip()
        elif sid is not None:
            label = f"说话人{int(sid) + 1}"
        else:
            label = "说话人1"
        entries.append({
            "speaker": label,
            "start": float(item.get("from", 0)),
            "end": float(item.get("to", item.get("from", 0))),
            "text": text,
        })
    return entries


def try_fetch_bilibili_subtitles(bvid: str, cid: int) -> list[dict]:
    tracks = list_subtitle_tracks(bvid, cid)
    if not tracks:
        print("B 站接口未返回字幕轨道（可能需要设置 SESSDATA）")
        return []

    for track in tracks:
        sub_url = track.get("subtitle_url") or track.get("url") or ""
        if not sub_url:
            continue
        if sub_url.startswith("//"):
            sub_url = "https:" + sub_url
        print("尝试字幕:", track.get("lan_doc") or track.get("lan") or sub_url)
        req = urllib.request.Request(
            sub_url,
            headers={**HEADERS, "Cookie": get_cookie_header()},
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
        entries = subtitle_entries_from_bilibili(payload)
        if entries:
            return entries
    return []


def diarize_audio(y: np.ndarray, sr: int, n_speakers: int) -> list[tuple[float, float, str]]:
    intervals = librosa.effects.split(y, top_db=28)
    feats, spans = [], []
    for start, end in intervals:
        if end - start < sr * 0.35:
            continue
        chunk = y[start:end]
        mfcc = librosa.feature.mfcc(y=chunk, sr=sr, n_mfcc=20)
        feats.append(mfcc.mean(axis=1))
        spans.append((start / sr, end / sr))

    if not spans:
        return [(0.0, len(y) / sr, "说话人1")]

    cluster_n = min(max(2, n_speakers), len(feats))
    if len(feats) == 1:
        labels = [0]
    else:
        labels = AgglomerativeClustering(
            n_clusters=cluster_n,
            linkage="average",
        ).fit_predict(np.array(feats))

    return [
        (start, end, f"说话人{label + 1}")
        for (start, end), label in zip(spans, labels)
    ]


def assign_speaker(seg_start: float, seg_end: float, diar: list[tuple[float, float, str]]) -> str:
    best, overlap = "说话人1", 0.0
    for ds, de, spk in diar:
        ov = max(0.0, min(seg_end, de) - max(seg_start, ds))
        if ov > overlap:
            overlap, best = ov, spk
    return best


def transcribe_with_diarization(
    wav_path: Path,
    model_name: str,
    n_speakers: int,
) -> list[dict]:
    y, sr = librosa.load(wav_path, sr=16000, mono=True)
    print(f"音频时长 {len(y) / sr:.1f}s，说话人聚类上限 {n_speakers}")
    diar = diarize_audio(y, sr, n_speakers)

    print(f"加载 Whisper 模型 {model_name} ...")
    model = whisper.load_model(model_name)
    print("转写中（CPU 上全片可能需十余分钟）...")
    result = model.transcribe(y, language="zh", verbose=False)

    entries = []
    for seg in result.get("segments") or []:
        text = str(seg.get("text", "")).strip()
        if not text:
            continue
        start = float(seg.get("start", 0))
        end = float(seg.get("end", start))
        entries.append({
            "speaker": assign_speaker(start, end, diar),
            "start": start,
            "end": end,
            "text": text,
        })
    return entries

In [ ]:
def group_entries_by_speaker(entries: list[dict]) -> dict[str, list[dict]]:
    grouped: dict[str, list[dict]] = defaultdict(list)
    for item in entries:
        grouped[item["speaker"]].append(item)
    for speaker in grouped:
        grouped[speaker].sort(key=lambda x: (x["start"], x["end"]))
    return dict(sorted(grouped.items(), key=lambda kv: kv[1][0]["start"] if kv[1] else 0))


def render_dialogue_text(
    title: str,
    bvid: str,
    grouped: dict[str, list[dict]],
    source: str,
) -> str:
    lines = [
        f"# {title}",
        f"BVID: {bvid}",
        f"来源: {source}",
        "",
    ]
    for speaker, segs in grouped.items():
        lines.append(f"## {speaker}")
        lines.append("")
        for seg in segs:
            lines.append(f"[{format_ts(seg['start'])} - {format_ts(seg['end'])}]")
            lines.append(seg["text"])
            lines.append("")
    return "\n".join(lines).rstrip() + "\n"


def save_dialogue_outputs(
    bvid: str,
    title: str,
    grouped: dict[str, list[dict]],
    source: str,
) -> tuple[Path, Path]:
    safe_name = re.sub(r"[\\/:*?\"<>|]", "_", title)[:80] or bvid
    txt_path = OUTPUT_DIR / f"{bvid}_{safe_name}.txt"
    json_path = OUTPUT_DIR / f"{bvid}_{safe_name}.json"

    txt_path.write_text(
        render_dialogue_text(title, bvid, grouped, source),
        encoding="utf-8",
    )
    json_path.write_text(
        json.dumps(
            {
                "bvid": bvid,
                "title": title,
                "source": source,
                "speakers": grouped,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    return txt_path, json_path

In [ ]:
meta = get_video_meta(BVID)
print("标题:", meta["title"])
print("时长:", meta["duration"], "秒")

entries = try_fetch_bilibili_subtitles(meta["bvid"], meta["cid"])
source = "bilibili_subtitle"

if not entries:
    with tempfile.TemporaryDirectory() as tmp:
        wav_path = Path(tmp) / f"{meta['bvid']}.wav"
        print("下载并转码音频...")
        download_audio_wav(
            meta["bvid"],
            meta["cid"],
            wav_path,
            max_duration_sec=MAX_DURATION_SEC,
        )
        entries = transcribe_with_diarization(
            wav_path,
            WHISPER_MODEL,
            N_SPEAKERS,
        )
    source = f"whisper_{WHISPER_MODEL}+diarization"

grouped = group_entries_by_speaker(entries)
txt_path, json_path = save_dialogue_outputs(
    meta["bvid"], meta["title"], grouped, source
)

print(f"说话人数: {len(grouped)}")
print(f"片段数: {len(entries)}")
print("已写入:")
print(" ", txt_path.resolve())
print(" ", json_path.resolve())

# 预览前 3 位说话人
for speaker, segs in list(grouped.items())[:3]:
    print("\n---", speaker, f"({len(segs)} 段) ---")
    for seg in segs[:3]:
        print(f"[{format_ts(seg['start'])}] {seg['text']}")